
# Bounding-Box Crops

Extract padded bounding-box crops per behavior label from VIA annotations.



In [1]:
import ast
import json
import time
from collections import defaultdict
from pathlib import Path
from typing import Any

import cv2
import pandas as pd
from tqdm import tqdm



In [2]:
# Configuration
CSV_PATH = Path("data/CBVD-5.csv")
IMG_ROOT = Path("data/labelframes/labelframes")
OUT_ROOT = Path("workdir/crops_raw")
PAD_FRACTION = 0.08
SKIP_ROWS = 9

# Behavior mapping from dataset metadata
BEHAVIOR_CODES = {
    0: "stand",
    1: "lying down",
    2: "foraging",
    3: "drinking water",
    4: "rumination",
}
BEHAVIOR_PRIORITY = [
    "drinking water",
    "foraging",
    "rumination",
    "lying down",
    "stand",
]




In [3]:
def parse_file_name(file_list_field: Any) -> str | None:
    """Extract first filename from VIA file_list field."""
    if isinstance(file_list_field, list):
        return file_list_field[0] if file_list_field else None

    if file_list_field is None:
        return None

    try:
        parsed = ast.literal_eval(str(file_list_field))
        if isinstance(parsed, list):
            return parsed[0] if parsed else None
    except (ValueError, SyntaxError):
        pass

    text = str(file_list_field).strip()
    return text or None


def parse_box(spatial_coordinates: Any) -> tuple[int, int, int, int]:
    """Extract (x, y, w, h) from VIA spatial coordinates."""
    coords = (
        json.loads(spatial_coordinates)
        if isinstance(spatial_coordinates, str)
        else spatial_coordinates
    )
    if not isinstance(coords, list) or not coords:
        raise ValueError("Invalid spatial_coordinates structure")

    if isinstance(coords[0], list):
        coords = coords[0]

    if len(coords) < 5:
        raise ValueError("Expected region shape + x,y,w,h")

    _, x, y, w, h = coords
    return int(x), int(y), int(w), int(h)


def extract_behavior(metadata: Any) -> str:
    """Extract canonical behavior label from VIA metadata."""
    try:
        if isinstance(metadata, str):
            try:
                meta_dict = json.loads(metadata)
            except json.JSONDecodeError:
                meta_dict = ast.literal_eval(metadata)
        elif isinstance(metadata, dict):
            meta_dict = metadata
        else:
            return "unknown"

        raw_codes = meta_dict.get("1") or meta_dict.get(1)
        if not raw_codes:
            return "unknown"

        codes = [
            int(token) for token in str(raw_codes).split(",") if token.strip().isdigit()
        ]
        labels = [BEHAVIOR_CODES[c] for c in codes if c in BEHAVIOR_CODES]

        for behavior in BEHAVIOR_PRIORITY:
            if behavior in labels:
                return behavior

        return labels[0] if labels else "unknown"
    except Exception:
        return "unknown"


def get_padded_crop_bounds(
    x: int,
    y: int,
    w: int,
    h: int,
    img_w: int,
    img_h: int,
    pad_fraction: float,
) -> tuple[int, int, int, int]:
    """Return bounded crop coordinates with padding."""
    pad_w = max(1, int(w * pad_fraction))
    pad_h = max(1, int(h * pad_fraction))

    x0 = max(0, x - pad_w)
    y0 = max(0, y - pad_h)
    x1 = min(img_w, x + w + pad_w)
    y1 = min(img_h, y + h + pad_h)

    return x0, y0, max(2, x1 - x0), max(2, y1 - y0)




In [4]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
if not IMG_ROOT.exists():
    raise FileNotFoundError(f"Image directory not found: {IMG_ROOT}")

OUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Processing {CSV_PATH} -> {OUT_ROOT}")

df = pd.read_csv(CSV_PATH, skiprows=SKIP_ROWS)
start_time = time.time()

image_groups: dict[str, list[tuple[int, int, int, int, str]]] = defaultdict(list)
skipped_rows = 0

for _, row in df.iterrows():
    try:
        img_name = parse_file_name(row["file_list"])
        if not img_name:
            skipped_rows += 1
            continue

        x, y, w, h = parse_box(row["spatial_coordinates"])
        behavior = extract_behavior(row["metadata"])
        image_groups[img_name].append((x, y, w, h, behavior))
    except Exception:
        skipped_rows += 1




Processing data/CBVD-5.csv -> workdir/crops_raw


In [5]:
crops_written = 0
missing_images = 0
invalid_crops = 0

for img_name, annotations in tqdm(image_groups.items(), desc="Cropping"):
    img_path = IMG_ROOT / img_name
    if not img_path.exists():
        missing_images += len(annotations)
        continue

    image = cv2.imread(str(img_path))
    if image is None:
        missing_images += len(annotations)
        continue

    img_h, img_w = image.shape[:2]

    for x, y, w, h, behavior in annotations:
        cx, cy, cw, ch = get_padded_crop_bounds(x, y, w, h, img_w, img_h, PAD_FRACTION)
        crop = image[cy : cy + ch, cx : cx + cw]
        if crop.size == 0 or crop.shape[0] < 2 or crop.shape[1] < 2:
            invalid_crops += 1
            continue

        behavior_dir = OUT_ROOT / behavior
        behavior_dir.mkdir(parents=True, exist_ok=True)

        stem = Path(img_name).stem
        crop_name = f"{stem}_{cx}_{cy}_{cw}_{ch}.jpg"
        crop_path = behavior_dir / crop_name

        if cv2.imwrite(str(crop_path), crop):
            crops_written += 1
        else:
            invalid_crops += 1




Cropping: 100%|██████████| 3199/3199 [00:32<00:00, 97.55it/s] 


In [6]:
processing_time = time.time() - start_time

behavior_counts: dict[str, int] = defaultdict(int)
for annotations in image_groups.values():
    for _, _, _, _, behavior in annotations:
        behavior_counts[behavior] += 1

print(f"Processed {len(image_groups)} images in {processing_time:.1f}s")
print(
    "Created "
    f"{crops_written} crops | "
    f"missing source boxes: {missing_images} | "
    f"invalid crops: {invalid_crops} | "
    f"skipped rows: {skipped_rows}"
)
print("Behavior distribution:")
for behavior, count in sorted(behavior_counts.items()):
    print(f"  {behavior}: {count}")
print(f"Output directory: {OUT_ROOT}")


Processed 3199 images in 33.5s
Created 25322 crops | missing source boxes: 0 | invalid crops: 2 | skipped rows: 0
Behavior distribution:
  drinking water: 744
  foraging: 5711
  lying down: 4518
  rumination: 6079
  stand: 8272
Output directory: workdir/crops_raw
